### Visualisation des 16 événements

#### Objectif

Je réalise une visualisation animée des 16 événements sélectionnés
dans le notebook 09 et sonifiés dans le notebook 10.

Chaque animation représente deux heures de données en 60 secondes.

#### Choix graphiques

- Carte : localisation géographique des stations et de la Senne.
- Cercles proportionnels : intensité et répartition des précipitations.
- Barres : concentration en oxygène dissous.
- Couleurs distinctes : différenciation des deux phénomènes.
- Indicateur temporel : suivi de l'évolution des données.

#### Méthode

Je développe une visualisation commune aux 16 événements,
avec des échelles et des paramètres graphiques identiques.

Cette visualisation servira à évaluer la détection et la localisation
des épisodes de faible oxygénation ainsi que la perception de la
répartition spatiale des précipitations.

Les versions avec et sans son seront visuellement identiques.

In [ ]:
# importation bibl

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

from pathlib import Path
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

Chargements des données

In [ ]:
dossier = Path("../data/totalité_juillet_septembre_2025")

# Charger les précipitations
df = pd.read_csv(
    dossier / "pluviometres_16_fusionnes.csv",
    parse_dates=["date"]
)

# Charger les données d'oxygène
stations_oxygene = ["Q01", "Q02", "Q03", "Q04"]

for station in stations_oxygene:

    oxygene = pd.read_csv(
        dossier / f"{station}.csv",
        sep="\t",
        encoding="cp1252",
        decimal=","
    )

    oxygene = oxygene.iloc[:, :2].copy()
    oxygene.columns = ["date", station]

    oxygene["date"] = pd.to_datetime(
        oxygene["date"].astype(str),
        format="%d.%m.%Y %H:%M:%S",
        errors="coerce"
    )

    oxygene[station] = pd.to_numeric(
        oxygene[station], errors="coerce"
    )

    df = df.merge(oxygene, on="date", how="left")

# Garder la période étudiée
df = df[
    (df["date"] >= "2025-07-01") &
    (df["date"] < "2025-10-01")
].sort_values("date").reset_index(drop=True)

# Identifier les pluviomètres
stations_pluie = [
    c for c in df.columns
    if c not in ["date"] + stations_oxygene
]

print("Pluviomètres :", len(stations_pluie))
print("Stations d'oxygène :", len(stations_oxygene))
print("Nombre de mesures :", len(df))

D'abord essaye avec un evenement

In [ ]:
selection = pd.read_csv(
    "../data/selection_stimuli.csv",
    parse_dates=["debut"]
)

numero = 1
duree = 60

debut = selection.iloc[numero - 1]["debut"]
fin = debut + pd.Timedelta(hours=2)

extrait = df[
    (df["date"] >= debut) &
    (df["date"] < fin)
].copy()

assert len(extrait) == 24

print("Événement :", numero)
print("Début :", debut)
print("Nombre de mesures :", len(extrait))

display(extrait.head())

#### Préparation des données géographiques

Je reprends les coordonnées des 
- 16 pluviomètres,
- 4 stations d'oxygène 
- le tracé de la senne   
utilisés dans le notebook 07.

In [ ]:
data = Path("../data")

# Charger les coordonnées des stations
coord_pluie = pd.read_csv(
    data / "coordonnees_pluviometres_flowbru.csv"
)

coord_oxygene = pd.read_csv(
    data / "coordonnees_oxygene_senne.csv"
)

# Charger le tracé de la Senne
senne = gpd.read_file(
    data / "tracé_senne.geojson"
).to_crs("EPSG:31370")

# Créer les points géographiques
points_pluie = gpd.GeoDataFrame(
    coord_pluie,
    geometry=gpd.points_from_xy(
        coord_pluie.x_lambert72,
        coord_pluie.y_lambert72
    ),
    crs="EPSG:31370"
)

points_oxygene = gpd.GeoDataFrame(
    coord_oxygene,
    geometry=gpd.points_from_xy(
        coord_oxygene.x_lambert72,
        coord_oxygene.y_lambert72
    ),
    crs="EPSG:31370"
)

print("Pluviomètres :", len(points_pluie))
print("Stations d'oxygène :", len(points_oxygene))

#### Première visualisation statique

Je visualise la localisation des stations et le tracé
de la Senne avant d'intégrer les données temporelles.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

fig.patch.set_facecolor("#FFFFFF")
ax.set_facecolor("#F5F5F2")

# Tracé de la Senne
senne.plot(
    ax=ax,
    color="#789BAF",
    linewidth=2
)

# Pluviomètres
points_pluie.plot(
    ax=ax,
    color="#469BCB",
    markersize=45,
    alpha=0.8,
    label="Pluviomètres"
)

# Stations d'oxygène
points_oxygene.plot(
    ax=ax,
    color="#35765C",
    marker="s",
    markersize=60,
    label="Stations d'oxygène"
)

# Noms des stations d'oxygène
for _, station in points_oxygene.iterrows():

    ax.annotate(
        station["capteur"],
        (station.geometry.x, station.geometry.y),
        xytext=(8, -10),
        textcoords="offset points",
        fontsize=10
    )

# Cadrage
ax.set_xlim(
    min(points_pluie.geometry.x.min(),
        points_oxygene.geometry.x.min()) - 1000,
    max(points_pluie.geometry.x.max(),
        points_oxygene.geometry.x.max()) + 1000
)

ax.set_ylim(
    min(points_pluie.geometry.y.min(),
        points_oxygene.geometry.y.min()) - 1000,
    max(points_pluie.geometry.y.max(),
        points_oxygene.geometry.y.max()) + 2500
)

ax.set_aspect("equal")
ax.set_title("Précipitations et oxygène dissous dans la Senne")
ax.legend(frameon=False)
ax.set_axis_off()

plt.tight_layout()
plt.show()

Vvérification pour préparer l'intégration des données :  
Cette vérification est nécessaire car les noms des pluviomètres dans les données des trois mois ne correspondent pas forcément à ceux utilisés dans le notebook 07.

In [ ]:
print("Colonnes des données de pluie :")
print(stations_pluie)

print("\nNoms dans le fichier géographique :")
print(coord_pluie["colonne_pluie_csv"].tolist())

#### Correspondance entre les données et les stations

J'harmonise les noms des pluviomètres afin d'associer
chaque mesure à sa position géographique correcte.

In [ ]:
import unicodedata

def harmoniser_nom(nom):
    nom = nom.strip().lower()
    nom = unicodedata.normalize("NFKD", nom)
    nom = "".join(c for c in nom if not unicodedata.combining(c))
    return nom.replace(".", "_").replace(";", "_")

# Correspondance avec les colonnes des données
correspondance = {
    harmoniser_nom(c): c for c in stations_pluie
}

points_pluie["colonne_donnees"] = (
    points_pluie["colonne_pluie_csv"]
    .apply(harmoniser_nom)
    .map(correspondance)
)

# Vérifier les correspondances
assert points_pluie["colonne_donnees"].notna().all()
assert points_pluie["colonne_donnees"].nunique() == 16

display(points_pluie[
    ["colonne_pluie_csv", "colonne_donnees"]
])

print("Les 16 pluviomètres sont correctement associés.")

Garder l'odre des coordonnées géographiques: rémarquer que les stations d'oxygène ont un nom numeroté qui ne correspond pas à l'odre de dispositino sur la carte

In [ ]:
# Extraire les valeurs dans le même ordre que les points géographiques
valeurs_pluie = extrait[
    points_pluie["colonne_donnees"].tolist()
].to_numpy(dtype=float)

valeurs_oxygene = extrait[
    points_oxygene["capteur"].tolist()
].to_numpy(dtype=float)

print("Dimensions pluie :", valeurs_pluie.shape)
print("Dimensions oxygène :", valeurs_oxygene.shape)

#### Premier essai graphique

Précipitations = cercles dont la
surface est proportionnelle aux quantités mesurées,  
Oxygène dissous = barres verticales.

Teste représentation statique avant de
développer l'animation.

##### Représentation de l'oxygène dissous

Je représente l'oxygène dissous par des jauges verticales
dont le remplissage est proportionnel à la concentration mesurée.

L'échelle provisoire est fixée de 0 à 12 mg/L.
Un trait horizontal indique le seuil critique de 3 mg/L.

Je conserve une couleur constante pour ne pas introduire
d'alerte visuelle supplémentaire dans l'expérience.

Les repères quantitatifs servent à faciliter la lecture
des précipitations et des concentrations d'oxygène.
Les échelles restent identiques pour les 16 événements.

In [ ]:
def dessiner_carte(i):

    # Échelles graphiques communes
    echelle_pluie = 800
    echelle_oxygene = 240
    maximum_oxygene = 10
    seuil_oxygene = 3
    largeur = 250

    fig, ax = plt.subplots(figsize=(10, 10))

    fig.patch.set_facecolor("white")
    ax.set_facecolor("#F5F5F2")

    # Fond géographique
    bruxelles.plot(
        ax=ax,
        facecolor="#F2F2EF",
        edgecolor="#C4C4C4",
        linewidth=0.8,
        zorder=0
    )

    # Tracé de la Senne
    senne.plot(ax=ax, color="#789BAF", linewidth=2)

    # Position fixe des pluviomètres
    ax.scatter(
        points_pluie.geometry.x,
        points_pluie.geometry.y,
        s=8,
        color="#A0A0A0"
    )

    # Cercles proportionnels aux précipitations
    pluie_i = valeurs_pluie[i]
    valides = np.isfinite(pluie_i) & (pluie_i > 0)

    ax.scatter(
        points_pluie.geometry.x[valides],
        points_pluie.geometry.y[valides],
        s=pluie_i[valides] * echelle_pluie,
        color="#469BCB",
        edgecolor="#2878A8",
        alpha=0.6
    )

    # Jauges d'oxygène
    for j, (_, station) in enumerate(points_oxygene.iterrows()):

        x = station.geometry.x
        y = station.geometry.y
        valeur = valeurs_oxygene[i, j]

        # Fond de la jauge
        ax.bar(
            x, maximum_oxygene * echelle_oxygene,
            bottom=y, width=largeur,
            color="#E5E7E9",
            edgecolor="#899398",
            zorder=4
        )

        # Remplissage selon la concentration
        if np.isfinite(valeur):
            ax.bar(
                x, valeur * echelle_oxygene,
                bottom=y, width=largeur,
                color="#35765C",
                zorder=5
            )

        # Repère du seuil de 3 mg/L
        ax.hlines(
            y + seuil_oxygene * echelle_oxygene,
            x - largeur * 0.7,
            x + largeur * 0.7,
            color="#B36C38",
            linewidth=1.5,
            zorder=6
        )

        # Identification de la station
        ax.annotate(
            station["capteur"],
            (x, y),
            xytext=(0, -14),
            textcoords="offset points",
            ha="center",
            fontsize=10
        )

    # Cadrage commun
    ax.set_xlim(
        min(points_pluie.geometry.x.min(),
            points_oxygene.geometry.x.min()) - 1000,
        max(points_pluie.geometry.x.max(),
            points_oxygene.geometry.x.max()) + 1000
    )

    ax.set_ylim(
        min(points_pluie.geometry.y.min(),
            points_oxygene.geometry.y.min()) - 1000,
        max(points_pluie.geometry.y.max(),
            points_oxygene.geometry.y.max()) + 2500
    )

    # Titre et indication temporelle
    ax.set_title(
        "Précipitations et oxygène dissous\n"
        + extrait.iloc[i]["date"].strftime("%d/%m/%Y — %H:%M"),
        fontsize=15,
        pad=20
    )

    ax.set_aspect("equal")
    ax.set_axis_off()

    # Légende des précipitations
    for mm in [0.1, 0.5, 1, 1.5]:
        ax.scatter(
            [], [],
            s=mm * echelle_pluie,
            color="#469BCB",
            edgecolor="#2878A8",
            alpha=0.6,
            label=f"{mm} mm"
        )

    ax.legend(
        title="Précipitations / 5 min",
        loc="upper left",
        frameon=False,
        labelspacing=2.5,
        handletextpad=2
    )

    # Légende de l'oxygène
    ax.text(
        0.02, 0.02,
        f"Oxygène dissous : 0–{maximum_oxygene} mg/L\n"
        "Trait orange : seuil de 3 mg/L",
        transform=ax.transAxes,
        fontsize=10,
        va="bottom"
    )

    plt.tight_layout()

    return fig

Trouver un instant intéressant à visualiser, avec le maximum de précipitations cumulées entre les stations :

In [ ]:
fig = dessiner_carte(12)
plt.show()

In [ ]:
i = np.argmax(np.nansum(valeurs_pluie, axis=1))

print("Instant à visualiser :", i)
print("Heure :", extrait.iloc[i]["date"])

#### Fond géographique

J'ajoute les limites de la Région de Bruxelles-Capitale
pour faciliter la localisation spatiale des stations.

In [ ]:
fichier_bruxelles = Path("../data/limites_bruxelles.geojson")

if not fichier_bruxelles.exists():

    url = (
        "https://bruxellesdata.opendatasoft.com/api/explore/v2.1/"
        "catalog/datasets/limites-administratives-des-communes-en-region-de-bruxelles-capitale/"
        "exports/geojson"
    )

    communes = gpd.read_file(url)

    bruxelles = communes[["geometry"]].dissolve()

    bruxelles.to_file(fichier_bruxelles, driver="GeoJSON")

bruxelles = gpd.read_file(fichier_bruxelles).to_crs("EPSG:31370")

#### Vérification des échelles graphiques

Je vérifie les valeurs extrêmes et les données manquantes
des 16 événements afin de définir des échelles communes.

In [ ]:
extraits_selection = pd.concat([
    df[
        (df["date"] >= debut) &
        (df["date"] < debut + pd.Timedelta(hours=2))
    ]
    for debut in selection["debut"]
])

for nom, colonnes in [
    ("Pluie", stations_pluie),
    ("Oxygène", stations_oxygene)
]:
    valeurs = extraits_selection[colonnes]

    print(nom)
    print("Minimum :", valeurs.min().min())
    print("Maximum :", valeurs.max().max())
    print("Valeurs manquantes :", valeurs.isna().sum().sum())
    print()

#### Animation du premier événement

J'anime les variations des précipitations et de l'oxygène
dissous sur une période de deux heures, compressée en
60 secondes, comme dans la sonification. La taille des cercles et la hauteur des barres évoluent progressivement entre chaque mesure afin de rendre l'animation plus fluide.

In [ ]:
# Paramètres de l'animation
duree = 60
fps = 25
nombre_images = duree * fps

temps_mesures = np.linspace(0, duree, len(extrait))
temps_animation = np.arange(nombre_images) / fps

# Interpolation des précipitations
pluie_animee = np.column_stack([
    np.interp(temps_animation, temps_mesures, colonne)
    for colonne in valeurs_pluie.T
])

# Interpolation de l'oxygène
oxygene_anime = np.column_stack([
    np.interp(temps_animation, temps_mesures, colonne)
    for colonne in valeurs_oxygene.T
])

print("Nombre d'images :", nombre_images)
print("Durée :", duree, "secondes")
print("Fréquence :", fps, "images/seconde")

#### Animation cercles et barres

In [ ]:
from matplotlib.collections import PathCollection
from matplotlib.patches import Rectangle

# Paramètres temporels
duree = 60
fps = 25
nombre_images = duree * fps

temps_mesures = np.linspace(0, duree, len(extrait))
temps_animation = np.arange(nombre_images) / fps

# Interpolation de la pluie
pluie_animee = np.column_stack([
    np.interp(temps_animation, temps_mesures, colonne)
    for colonne in valeurs_pluie.T
])

# Interpolation de l'oxygène
oxygene_animee = np.column_stack([
    np.interp(temps_animation, temps_mesures, colonne)
    for colonne in valeurs_oxygene.T
])

# Reprendre la carte
fig = dessiner_carte(0)
ax = fig.axes[0]

collections = [
    c for c in ax.collections
    if isinstance(c, PathCollection)
]

cercles = collections[1]

cercles.set_offsets(np.column_stack([
    points_pluie.geometry.x,
    points_pluie.geometry.y
]))

jauges = [
    p for p in ax.patches
    if isinstance(p, Rectangle) and p.get_zorder() == 5
]

assert len(jauges) == 4

# Actualisation des éléments graphiques
def actualiser(i):

    cercles.set_sizes(pluie_animee[i] * 800)

    for j, jauge in enumerate(jauges):
        jauge.set_height(oxygene_animee[i, j] * 240)

    instant = extrait.iloc[0]["date"] + pd.Timedelta(
        seconds=temps_animation[i] / duree * 115 * 60
    )

    ax.set_title(
        "Précipitations et oxygène dissous\n"
        + instant.strftime("%d/%m/%Y — %H:%M"),
        fontsize=15,
        pad=20
    )

    return [cercles] + jauges + [ax.title]

# Vérifier le fonctionnement avant le rendu
actualiser(0)
actualiser(nombre_images - 1)
actualiser(0)

# Créer l'animation
animation = FuncAnimation(
    fig,
    actualiser,
    frames=nombre_images,
    interval=1000 / fps,
    repeat=False
)

plt.close(fig)

print("Animation prête et vérifiée.")

Visualisation

In [ ]:
import imageio_ffmpeg
import matplotlib as mpl

mpl.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()

display(HTML(animation.to_html5_video()))

#### Vérification de la visualisation sur un autre événement

Je vérifie que les encodages graphiques et les échelles
retenues restent adaptés à différents événements,
avec et sans épisode de faible oxygénation.

In [ ]:
numero = 9

debut = selection.iloc[numero - 1]["debut"]
fin = debut + pd.Timedelta(hours=2)

extrait = df[
    (df["date"] >= debut) &
    (df["date"] < fin)
].copy()

assert len(extrait) == 24

valeurs_pluie = extrait[
    points_pluie["colonne_donnees"].tolist()
].to_numpy(dtype=float)

valeurs_oxygene = extrait[
    points_oxygene["capteur"].tolist()
].to_numpy(dtype=float)

fig = dessiner_carte(12)
plt.show()

Reprise du code d'animation et visualisation precedentes

In [ ]:
from matplotlib.collections import PathCollection
from matplotlib.patches import Rectangle

# Paramètres temporels
duree = 60
fps = 25
nombre_images = duree * fps

temps_mesures = np.linspace(0, duree, len(extrait))
temps_animation = np.arange(nombre_images) / fps

# Interpolation de la pluie
pluie_animee = np.column_stack([
    np.interp(temps_animation, temps_mesures, colonne)
    for colonne in valeurs_pluie.T
])

# Interpolation de l'oxygène
oxygene_animee = np.column_stack([
    np.interp(temps_animation, temps_mesures, colonne)
    for colonne in valeurs_oxygene.T
])

# Reprendre la carte
fig = dessiner_carte(0)
ax = fig.axes[0]

collections = [
    c for c in ax.collections
    if isinstance(c, PathCollection)
]

cercles = collections[1]

cercles.set_offsets(np.column_stack([
    points_pluie.geometry.x,
    points_pluie.geometry.y
]))

jauges = [
    p for p in ax.patches
    if isinstance(p, Rectangle) and p.get_zorder() == 5
]

assert len(jauges) == 4

# Actualisation des éléments graphiques
def actualiser(i):

    cercles.set_sizes(pluie_animee[i] * 800)

    for j, jauge in enumerate(jauges):
        jauge.set_height(oxygene_animee[i, j] * 240)

    instant = extrait.iloc[0]["date"] + pd.Timedelta(
        seconds=temps_animation[i] / duree * 115 * 60
    )

    ax.set_title(
        "Précipitations et oxygène dissous\n"
        + instant.strftime("%d/%m/%Y — %H:%M"),
        fontsize=15,
        pad=20
    )

    return [cercles] + jauges + [ax.title]

# Vérifier le fonctionnement avant le rendu
actualiser(0)
actualiser(nombre_images - 1)
actualiser(0)

# Créer l'animation
animation = FuncAnimation(
    fig,
    actualiser,
    frames=nombre_images,
    interval=1000 / fps,
    repeat=False
)

plt.close(fig)

print("Animation prête et vérifiée.")

In [ ]:
import imageio_ffmpeg
import matplotlib as mpl

mpl.rcParams["animation.ffmpeg_path"] = imageio_ffmpeg.get_ffmpeg_exe()

display(HTML(animation.to_html5_video()))

Dans cette visualisation il y à que des petits points gris pour indiquer les pluviometres et 

In [ ]:
pluie_9 = df.loc[
    (df["date"] >= "2025-07-10 12:00") &
    (df["date"] < "2025-07-10 14:00"),
    stations_pluie
].to_numpy(dtype=float)

print("Précipitations maximales :", np.nanmax(pluie_9))
print("Nombre de mesures positives :", np.sum(pluie_9 > 0))
print("Nombre de valeurs manquantes :", np.isnan(pluie_9).sum())

#### Automatisation de la visualisation des 16 événements

Je généralise la méthode de visualisation à l'ensemble
des événements sélectionnés, en conservant les mêmes
encodages graphiques, les mêmes échelles et le même cadrage.

In [ ]:
def charger_evenement(numero):

    debut = selection.iloc[numero - 1]["debut"]
    fin = debut + pd.Timedelta(hours=2)

    extrait = df[
        (df["date"] >= debut) &
        (df["date"] < fin)
    ].copy()

    assert len(extrait) == 24

    pluie = extrait[
        points_pluie["colonne_donnees"].tolist()
    ].to_numpy(dtype=float)

    oxygene = extrait[
        points_oxygene["capteur"].tolist()
    ].to_numpy(dtype=float)

    return extrait, pluie, oxygene

Vérification des 16 événements

In [ ]:
for numero in range(1, 17):

    extrait, valeurs_pluie, valeurs_oxygene = charger_evenement(numero)

    fig = dessiner_carte(12)

    fig.suptitle(f"Événement {numero}", fontsize=12)

    plt.show()
    plt.close(fig)

Ici en heaut il y a 16 cartes statiques correspondant au milieu de chaque fenêtre temporelle. J'ai choisi de vérifier d'abord une image par événement pour éviter de générer inutilement 24 000 images avant la validation finale.

#### Généralisation de l'animation aux 16 événements

Je crée une fonction réutilisable permettant de générer
une animation pour chaque événement sélectionné,
en conservant les mêmes paramètres graphiques et temporels.

In [ ]:
def creer_animation(numero):

    global extrait, valeurs_pluie, valeurs_oxygene

    extrait, valeurs_pluie, valeurs_oxygene = charger_evenement(numero)

    duree = 60
    fps = 25
    nombre_images = duree * fps

    temps_mesures = np.linspace(0, duree, len(extrait))
    temps_animation = np.arange(nombre_images) / fps

    # Interpolation
    pluie_animee = np.column_stack([
        np.interp(temps_animation, temps_mesures, colonne)
        for colonne in valeurs_pluie.T
    ])

    oxygene_animee = np.column_stack([
        np.interp(temps_animation, temps_mesures, colonne)
        for colonne in valeurs_oxygene.T
    ])

    # Création de la carte
    fig = dessiner_carte(0)
    ax = fig.axes[0]

    collections = [
        c for c in ax.collections
        if isinstance(c, PathCollection)
    ]

    cercles = collections[1]

    cercles.set_offsets(np.column_stack([
        points_pluie.geometry.x,
        points_pluie.geometry.y
    ]))

    jauges = [
        p for p in ax.patches
        if isinstance(p, Rectangle) and p.get_zorder() == 5
    ]

    assert len(jauges) == 4

    date_debut = extrait.iloc[0]["date"]

    # Actualisation de l'animation
    def actualiser(i):

        cercles.set_sizes(pluie_animee[i] * 800)

        for j, jauge in enumerate(jauges):
            jauge.set_height(oxygene_animee[i, j] * 240)

        instant = date_debut + pd.Timedelta(
            seconds=temps_animation[i] / duree * 115 * 60
        )

        ax.set_title(
            "Précipitations et oxygène dissous\n"
            + instant.strftime("%d/%m/%Y — %H:%M"),
            fontsize=15,
            pad=20
        )

        return [cercles] + jauges + [ax.title]

    animation = FuncAnimation(
        fig,
        actualiser,
        frames=nombre_images,
        interval=1000 / fps,
        repeat=False
    )

    plt.close(fig)

    return animation

Tester la fonction :  
J'ai choisi l'événement 3 pour tester une animation différente de celles déjà observées. Je conserve le design et la convention temporelle provisoire.

In [ ]:
animation = creer_animation(3)

display(HTML(animation.to_html5_video()))